# ML Engine V1 — Standard LightGBM vs Focal Loss

Run this notebook from the project directory containing `train_m1_m4.parquet` and `calib_m5.parquet`.

M1-M4 is split chronologically into an internal training/validation portion for early stopping. M5 is then used for the Standard-vs-Focal comparison. M6 remains untouched.

## 1. Imports and Package Setup

In [5]:
import sys
import subprocess
import os
import json
import warnings

def install_requirements():
    required_packages = [
        'pandas',
        'numpy',
        'lightgbm',
        'scikit-learn',
        'joblib',
        'pyarrow',
        'fastparquet',
        'scipy'
    ]

    for package in required_packages:
        try:
            if package == 'scikit-learn':
                __import__('sklearn')
            else:
                __import__(package)
        except ImportError:
            print(f"Package '{package}' not found. Installing")
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", package]
            )

    print("All dependencies installed.\n")


install_requirements()

import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib

from scipy.special import expit

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve
)


# CONFIGURATION
TRAIN_PATH = "train_m1_m4.parquet"
CALIB_PATH = "calib_m5.parquet"

BASELINE_MODEL_PATH = "baseline_lightgbm.pkl"
FOCAL_MODEL_PATH = "focal_lightgbm.pkl"
WINNER_MODEL_PATH = "sentinel_x_winner.pkl"

BASELINE_PREDICTION_PATH = "baseline_m5_predictions.csv"
FOCAL_PREDICTION_PATH = "focal_m5_predictions.csv"
WINNER_PREDICTION_PATH = "m5_winning_predictions.csv"

BASELINE_FEATURE_IMP_PATH = "baseline_feature_importance.csv"
FOCAL_FEATURE_IMP_PATH = "focal_feature_importance.csv"
WINNER_FEATURE_IMP_PATH = "sentinel_x_feature_importance.csv"

RESULTS_PATH = "ml_engine_results.json"

FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0

INTERNAL_VALIDATION_FRACTION = 0.15




All dependencies installed.



## 2. Configuration

In [21]:
# CONFIGURATION
TRAIN_PATH = "train_m1_m4.parquet"
CALIB_PATH = "calib_m5.parquet"

BASELINE_MODEL_PATH = "baseline_lightgbm.pkl"
FOCAL_MODEL_PATH = "focal_lightgbm.pkl"
WINNER_MODEL_PATH = "FinanceRiskManager_winner.pkl"

BASELINE_PREDICTION_PATH = "baseline_m5_predictions.csv"
FOCAL_PREDICTION_PATH = "focal_m5_predictions.csv"
WINNER_PREDICTION_PATH = "m5_winning_predictions.csv"

BASELINE_FEATURE_IMP_PATH = "baseline_feature_importance.csv"
FOCAL_FEATURE_IMP_PATH = "focal_feature_importance.csv"
WINNER_FEATURE_IMP_PATH = "FinanceRiskManager_feature_importance.csv"

RESULTS_PATH = "ml_engine_results.json"

FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0

INTERNAL_VALIDATION_FRACTION = 0.15

PREPROCESSING_ARTIFACT_PATH = "preprocessing_artifacts.pkl"


## 3. Data Loading

In [22]:
# ==========================================
# DATA LOADING
# ==========================================

def load_data():
    """
    Loads the M1-M4 training parquet and the M5 calibration parquet.
    M6 (real + stress) is intentionally NOT loaded here -- it stays
    untouched until the held-out evaluation notebook
    (phase3_held_out_evaluation.ipynb) scores the already-frozen model.
    """

    if not os.path.exists(TRAIN_PATH):
        raise FileNotFoundError(
            f"Training file missing: {TRAIN_PATH}. "
            f"Run dataB(generated).py first."
        )

    if not os.path.exists(CALIB_PATH):
        raise FileNotFoundError(
            f"Calibration file missing: {CALIB_PATH}. "
            f"Run dataB(generated).py first."
        )

    print(f"Loading training data: {TRAIN_PATH}")
    train_df = pd.read_parquet(TRAIN_PATH)

    print(f"Loading calibration data: {CALIB_PATH}")
    calib_df = pd.read_parquet(CALIB_PATH)

    print(
        f"Loaded Train: {len(train_df):,} rows | "
        f"Calib: {len(calib_df):,} rows"
    )

    return train_df, calib_df


In [23]:
train_df, calib_df = load_data()

Loading training data: train_m1_m4.parquet
Loading calibration data: calib_m5.parquet
Loaded Train: 41,337 rows | Calib: 8,858 rows


## 4. Internal Temporal Validation Split

In [24]:
train_df = train_df.sort_values(
    'TransactionDT'
).reset_index(drop=True)

internal_val_start = int(
    len(train_df)
    * (1.0 - INTERNAL_VALIDATION_FRACTION)
)

train_core_df = train_df.iloc[
    :internal_val_start
].copy()

internal_val_df = train_df.iloc[
    internal_val_start:
].copy()

print(
    f"Internal Temporal Split -> "
    f"Train: {len(train_core_df):,} | "
    f"Validation: {len(internal_val_df):,}"
)

Internal Temporal Split -> Train: 35,136 | Validation: 6,201


## 5. Temporal Features and Feature Preparation

In [25]:
# ==========================================
# TEMPORAL FEATURES
# ==========================================

def add_temporal_features(df):
    df = df.copy()

    if 'TransactionDT' not in df.columns:
        raise ValueError(
            "TransactionDT column missing."
        )

    # TransactionDT is measured in seconds.
    df['TransactionHour'] = (
        (df['TransactionDT'] // 3600) % 24
    ).astype(np.int8)

    df['TransactionDay'] = (
        df['TransactionDT'] // (24 * 3600)
    ).astype(np.int32)

    df['HourSin'] = np.sin(
        2 * np.pi * df['TransactionHour'] / 24
    ).astype(np.float32)

    df['HourCos'] = np.cos(
        2 * np.pi * df['TransactionHour'] / 24
    ).astype(np.float32)

    # Raw absolute timestamp is removed to reduce
    # memorization of exact temporal position.
    df.drop(
        columns=['TransactionDT'],
        inplace=True
    )

    return df


# ==========================================
# FEATURE PREPARATION
# ==========================================

def prepare_features(
    train_df,
    internal_val_df,
    calib_df
):

    print("\nPreparing Model Features & Temporal Transformations")

    train_df = add_temporal_features(train_df)
    internal_val_df = add_temporal_features(internal_val_df)
    calib_df = add_temporal_features(calib_df)

    target = 'isFraud'

    y_train = train_df[target].astype(np.int8)
    y_internal_val = internal_val_df[target].astype(np.int8)
    y_calib = calib_df[target].astype(np.int8)

    # Drop identifiers to force behavioral learning rather than memorization.
    # card1 is intentionally retained because it is a meaningful
    # transaction entity and also supports the engineered card features.
    excluded_columns = [
        'isFraud',
        'TransactionID',
        'device_id',
        'ip_address'
    ]

    feature_columns = [
        col
        for col in train_df.columns
        if col not in excluded_columns
    ]

    X_train = train_df[feature_columns].copy()
    X_internal_val = internal_val_df[feature_columns].copy()
    X_calib = calib_df[feature_columns].copy()

    if list(X_train.columns) != list(X_internal_val.columns):
        raise ValueError(
            "Training and internal validation feature columns do not match."
        )

    if list(X_train.columns) != list(X_calib.columns):
        raise ValueError(
            "Training and calibration feature columns do not match."
        )

    # Known identifier-like numerical columns are represented as
    # categorical variables instead of continuous numbers.
    categorical_id_columns = [
        'card1',
        'card2',
        'card3',
        'card5',
        'addr1',
        'addr2'
    ]

    categorical_columns = []
    category_maps = {}

    for col in feature_columns:

        is_existing_category = (
            isinstance(
                X_train[col].dtype,
                pd.CategoricalDtype
            )
        )

        is_string_feature = (
            pd.api.types.is_object_dtype(X_train[col])
            or pd.api.types.is_string_dtype(X_train[col])
        )

        should_be_categorical = (
            is_existing_category
            or is_string_feature
            or col in categorical_id_columns
        )

        if should_be_categorical:

            categorical_columns.append(col)

            train_values = X_train[col].astype('string')
            val_values = X_internal_val[col].astype('string')
            calib_values = X_calib[col].astype('string')

            # Categories are learned from training data only.
            categories = pd.Index(
                train_values.dropna().unique()
            )

            X_train[col] = pd.Categorical(
                train_values,
                categories=categories
            )

            X_internal_val[col] = pd.Categorical(
                val_values,
                categories=categories
            )

            X_calib[col] = pd.Categorical(
                calib_values,
                categories=categories
            )

            # Persist the exact train-derived category list so the
            # held-out M6 evaluation encodes test data identically
            # instead of re-deriving categories from test data.
            category_maps[col] = categories.tolist()

    print(
        f"Total features: {len(feature_columns)} | "
        f"Categorical: {len(categorical_columns)}"
    )

    print(
        f"Training Fraud Rate: "
        f"{y_train.mean() * 100:.2f}%"
    )

    print(
        f"Internal Validation Fraud Rate: "
        f"{y_internal_val.mean() * 100:.2f}%"
    )

    print(
        f"Calibration Fraud Rate: "
        f"{y_calib.mean() * 100:.2f}%"
    )

    return (
        X_train,
        y_train,
        X_internal_val,
        y_internal_val,
        X_calib,
        y_calib,
        feature_columns,
        categorical_columns,
        category_maps
    )




In [26]:
(
    X_train,
    y_train,
    X_internal_val,
    y_internal_val,
    X_calib,
    y_calib,
    feature_columns,
    categorical_features,
    category_maps
) = prepare_features(
    train_core_df,
    internal_val_df,
    calib_df
)


Preparing Model Features & Temporal Transformations


C:\Users\saket\AppData\Local\Temp\ipykernel_29012\3211577208.py:144: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_internal_val[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_29012\3211577208.py:149: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_calib[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_29012\3211577208.py:144: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_internal_val[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_29012\3211577208.py:149: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-n

Total features: 440 | Categorical: 35
Training Fraud Rate: 3.37%
Internal Validation Fraud Rate: 4.27%
Calibration Fraud Rate: 3.17%


C:\Users\saket\AppData\Local\Temp\ipykernel_29012\3211577208.py:149: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_calib[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_29012\3211577208.py:144: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_internal_val[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_29012\3211577208.py:149: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_calib[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_29012\3211577208.py:144: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null ent

## 6. Focal Loss and Evaluation Functions

In [33]:
# ==========================================
# PR-AUC EVALUATION FOR LIGHTGBM
# ==========================================

def pr_auc_eval_standard(preds, train_data):

    labels = train_data.get_label()

    score = average_precision_score(
        labels,
        preds
    )

    return (
        'pr_auc',
        float(score),
        True
    )


def pr_auc_eval_focal(preds, train_data):

    labels = train_data.get_label()

    probabilities = expit(preds)

    score = average_precision_score(
        labels,
        probabilities
    )

    return (
        'pr_auc',
        float(score),
        True
    )


# ==========================================
# CUSTOM FOCAL LOSS MATHEMATICS
# ==========================================

def focal_loss_lgb(preds, train_data):

    labels = train_data.get_label()

    alpha = FOCAL_ALPHA
    gamma = FOCAL_GAMMA

    p = expit(preds)

    eps = 1e-9

    p = np.clip(
        p,
        eps,
        1.0 - eps
    )

    q = 1.0 - p

    # Positive-class gradient and Hessian
    positive_b = (
        gamma * p * np.log(p)
        - q
    )

    positive_grad = (
        alpha
        * (q ** gamma)
        * positive_b
    )

    positive_hess = (
        alpha
        * p
        * (q ** gamma)
        * (
            gamma * np.log(p) * (q - gamma * p)
            + (2.0 * gamma + 1.0) * q
        )
    )

    # Negative-class gradient and Hessian
    negative_d = (
        p
        -
        gamma * q * np.log(q)
    )

    negative_grad = (
        (1.0 - alpha)
        * (p ** gamma)
        * negative_d
    )

    negative_hess = (
        (1.0 - alpha)
        * (p ** gamma)
        * q
        * (
            gamma * np.log(q) * (p - gamma * q)
            + (2.0 * gamma + 1.0) * p
        )
    )

    grad = np.where(
        labels == 1,
        positive_grad,
        negative_grad
    )

    hess = np.where(
        labels == 1,
        positive_hess,
        negative_hess
    )

    # LightGBM needs a numerically stable positive Hessian.
    hess = np.maximum(
        hess,
        1e-8
    )

    return grad, hess


def focal_loss_eval(preds, train_data):

    labels = train_data.get_label()

    alpha = FOCAL_ALPHA
    gamma = FOCAL_GAMMA

    probabilities = expit(preds)

    probabilities = np.clip(
        probabilities,
        1e-9,
        1.0 - 1e-9
    )

    loss = (
        -alpha
        * ((1.0 - probabilities) ** gamma)
        * np.log(probabilities)
        * labels

        -

        (1.0 - alpha)
        * (probabilities ** gamma)
        * np.log(1.0 - probabilities)
        * (1.0 - labels)
    )

    return (
        'focal_loss',
        float(np.mean(loss)),
        False
    )


# ==========================================
# MODEL METRIC EVALUATION
# ==========================================

def find_best_f1_threshold(
    y_true,
    probabilities
):

    precisions, recalls, thresholds = (
        precision_recall_curve(
            y_true,
            probabilities
        )
    )

    if len(thresholds) == 0:
        return 0.5, 0.0

    f1_scores = (
        2.0
        * precisions[:-1]
        * recalls[:-1]
        /
        (
            precisions[:-1]
            + recalls[:-1]
            + 1e-9
        )
    )

    best_idx = np.argmax(
        f1_scores
    )

    return (
        float(thresholds[best_idx]),
        float(f1_scores[best_idx])
    )


def evaluate_model(
    name,
    y_true,
    probabilities
):

    pr_auc = average_precision_score(
        y_true,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_true,
        probabilities
    )

    # Default threshold is reported only as a reference.
    predictions_05 = (
        probabilities >= 0.5
    ).astype(np.int8)

    precision_05 = precision_score(
        y_true,
        predictions_05,
        zero_division=0
    )

    recall_05 = recall_score(
        y_true,
        predictions_05,
        zero_division=0
    )

    f1_05 = f1_score(
        y_true,
        predictions_05,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions_05
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    # Diagnostic only.
    # This threshold is NOT used to choose the winner.
    best_threshold, max_f1 = (
        find_best_f1_threshold(
            y_true,
            probabilities
        )
    )

    print(
        f"\n--- {name} Results (M5 Calibration) ---"
    )

    print(
        f"PR-AUC (Primary):       {pr_auc:.6f}"
    )

    print(
        f"ROC-AUC:                {roc_auc:.6f}"
    )

    print(
        f"Precision @ 0.5:        {precision_05:.6f}"
    )

    print(
        f"Recall @ 0.5:           {recall_05:.6f}"
    )

    print(
        f"F1 @ 0.5:               {f1_05:.6f}"
    )

    print(
        f"False Positives @ 0.5:  {fp:,}"
    )

    print(
        f"False Negatives @ 0.5:  {fn:,}"
    )

    print(
        f"False Positive Rate:    {fpr:.6f}"
    )

    print(
        f"Max F1-Score:           {max_f1:.6f}"
    )

    print(
        f"Best F1 Threshold:      {best_threshold:.6f}"
    )

    print("\nConfusion Matrix:")
    print(
        f"TN: {tn:,} | FP: {fp:,}"
    )
    print(
        f"FN: {fn:,} | TP: {tp:,}"
    )

    return {
        'pr_auc': float(pr_auc),
        'roc_auc': float(roc_auc),
        'precision_at_0.5': float(precision_05),
        'recall_at_0.5': float(recall_05),
        'f1_at_0.5': float(f1_05),
        'false_positives_at_0.5': int(fp),
        'false_negatives_at_0.5': int(fn),
        'true_positives_at_0.5': int(tp),
        'true_negatives_at_0.5': int(tn),
        'false_positive_rate': float(fpr),
        'max_f1': float(max_f1),
        'best_f1_threshold': float(best_threshold)
    }




## 7. Train Standard LightGBM Baseline

In [28]:
# ==========================================
# COMMON LIGHTGBM PARAMETERS
# ==========================================

def get_common_parameters():

    return {
        'learning_rate': 0.05,

        'num_leaves': 31,

        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,

        'min_child_samples': 50,

        'lambda_l1': 0.1,
        'lambda_l2': 1.0,

        'verbosity': -1,
        'random_state': 42,
        'n_jobs': -1,

        # Disable LightGBM's default metric because
        # PR-AUC is our experiment metric.
        'metric': 'None'
    }


# ==========================================
# MODEL A: STANDARD LIGHTGBM
# ==========================================

def train_baseline_model(
    X_train,
    y_train,
    X_internal_val,
    y_internal_val,
    categorical_features
):

    print("\n--- Training Model A: Standard LightGBM ---")

    train_data = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=categorical_features,
        free_raw_data=False
    )

    internal_val_data = lgb.Dataset(
        X_internal_val,
        label=y_internal_val,
        categorical_feature=categorical_features,
        reference=train_data,
        free_raw_data=False
    )

    params = get_common_parameters()

    params.update({
        'objective': 'binary',

        # Dynamic training ratio instead of hard-coded 25.
        'scale_pos_weight': (
            (y_train == 0).sum()
            /
            (y_train == 1).sum()
        )
    })

    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,

        valid_sets=[
            internal_val_data
        ],

        valid_names=[
            'InternalValidation'
        ],

        feval=pr_auc_eval_standard,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=50,
                first_metric_only=True,
                verbose=False
            )
        ]
    )

    print(
        f"Standard LightGBM Best Iteration: "
        f"{model.best_iteration}"
    )

    print(
        f"Standard LightGBM Class Weight: "
        f"{params['scale_pos_weight']:.4f}"
    )

    return model


# ==========================================
# MODEL B: FOCAL LOSS LIGHTGBM
# ==========================================

def train_focal_model(
    X_train,
    y_train,
    X_internal_val,
    y_internal_val,
    categorical_features
):

    print(
        "\n--- Training Model B: Custom Focal-Loss LightGBM ---"
    )

    train_data = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=categorical_features,
        free_raw_data=False
    )

    internal_val_data = lgb.Dataset(
        X_internal_val,
        label=y_internal_val,
        categorical_feature=categorical_features,
        reference=train_data,
        free_raw_data=False
    )

    params = get_common_parameters()

    # Custom objective is supplied as a callable.
    params['objective'] = focal_loss_lgb

    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,

        valid_sets=[
            internal_val_data
        ],

        valid_names=[
            'InternalValidation'
        ],

        feval=[
            pr_auc_eval_focal,
            focal_loss_eval
        ],

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=50,
                first_metric_only=True,
                verbose=False
            )
        ]
    )

    print(
        f"Focal Loss Best Iteration: "
        f"{model.best_iteration}"
    )

    print(
        f"Focal Loss Parameters: "
        f"alpha={FOCAL_ALPHA}, "
        f"gamma={FOCAL_GAMMA}"
    )

    return model


# ==========================================
# FEATURE IMPORTANCE
# ==========================================

def save_feature_importance(
    model,
    feature_columns,
    filename
):

    print(
        f"\nSaving feature importance to: "
        f"{filename}"
    )

    importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': model.feature_importance(
            importance_type='gain'
        )
    })

    importance = importance.sort_values(
        'Importance',
        ascending=False
    )

    print(
        importance.head(30).to_string(
            index=False
        )
    )

    importance.to_csv(
        filename,
        index=False
    )





# ==========================================
# SAVE TRAINED MODEL + PREDICTIONS
# ==========================================

def save_model_outputs(
    model,
    predictions,
    y_true,
    model_path,
    predictions_path
):
    """
    Persists a trained LightGBM model (joblib) and its M5 calibration
    predictions (CSV: actual_fraud, fraud_probability) so downstream
    notebooks (phase2 cost optimizer, phase3 held-out evaluation) can
    load them without retraining.
    """

    print(f"\nSaving model to: {model_path}")
    joblib.dump(model, model_path)

    predictions_df = pd.DataFrame({
        'actual_fraud': np.asarray(y_true),
        'fraud_probability': np.asarray(predictions)
    })

    print(f"Saving predictions to: {predictions_path}")
    predictions_df.to_csv(predictions_path, index=False)


# ==========================================
# SAVE PREPROCESSING ARTIFACTS
# ==========================================

def save_preprocessing_artifacts(
    feature_columns,
    categorical_features,
    category_maps,
    winning_model_name,
    artifact_path
):
    """
    Persists everything needed to featurize a brand-new dataset (M6
    real / M6 stress) EXACTLY the way training data was featurized:
    the ordered feature column list, which columns are categorical,
    and the train-derived category vocabulary for each of them (so
    test-time categories never leak from test data itself). Also
    records which model won, so the held-out evaluation notebook
    knows whether to apply expit() to raw predictions.
    """

    print(f"\nSaving preprocessing artifacts to: {artifact_path}")

    artifacts = {
        'feature_columns': feature_columns,
        'categorical_features': categorical_features,
        'category_maps': category_maps,
        'winning_model_name': winning_model_name
    }

    joblib.dump(artifacts, artifact_path)


## 8. Train and Evaluate Both Models on M5

In [29]:
model_a = train_baseline_model(
    X_train,
    y_train,
    X_internal_val,
    y_internal_val,
    categorical_features
)

preds_a = model_a.predict(
    X_calib,
    num_iteration=model_a.best_iteration
)

metrics_a = evaluate_model(
    "Model A (Standard LightGBM)",
    y_calib,
    preds_a
)

model_b = train_focal_model(
    X_train,
    y_train,
    X_internal_val,
    y_internal_val,
    categorical_features
)

raw_preds_b = model_b.predict(
    X_calib,
    num_iteration=model_b.best_iteration
)

preds_b = expit(raw_preds_b)

metrics_b = evaluate_model(
    "Model B (Focal Loss)",
    y_calib,
    preds_b
)


--- Training Model A: Standard LightGBM ---
Standard LightGBM Best Iteration: 276
Standard LightGBM Class Weight: 28.6506

--- Model A (Standard LightGBM) Results (M5 Calibration) ---
PR-AUC (Primary):       0.405820
ROC-AUC:                0.847339
Precision @ 0.5:        0.418685
Recall @ 0.5:           0.430605
F1 @ 0.5:               0.424561
False Positives @ 0.5:  168
False Negatives @ 0.5:  160
False Positive Rate:    0.019587
Max F1-Score:           0.444444
Best F1 Threshold:      0.693890

Confusion Matrix:
TN: 8,409 | FP: 168
FN: 160 | TP: 121

--- Training Model B: Custom Focal-Loss LightGBM ---
Focal Loss Best Iteration: 119
Focal Loss Parameters: alpha=0.75, gamma=2.0

--- Model B (Focal Loss) Results (M5 Calibration) ---
PR-AUC (Primary):       0.409109
ROC-AUC:                0.856829
Precision @ 0.5:        0.640845
Recall @ 0.5:           0.323843
F1 @ 0.5:               0.430260
False Positives @ 0.5:  51
False Negatives @ 0.5:  190
False Positive Rate:    0.005946


## 9. Compare Models

In [30]:
print("\n==================================================")
print("VERDICT")
print("==================================================")

score_a = metrics_a['pr_auc']
score_b = metrics_b['pr_auc']

if score_b > score_a:
    difference = score_b - score_a
    print(
        f"WINNER: Focal Loss by +{difference:.6f} PR-AUC."
    )
    winner = "Focal_Loss_LightGBM"
    winner_model = model_b
    winner_predictions = preds_b
    winner_metrics = metrics_b
else:
    difference = score_a - score_b
    print(
        f"WINNER: Standard LightGBM by +{difference:.6f} PR-AUC."
    )
    winner = "Standard_LightGBM"
    winner_model = model_a
    winner_predictions = preds_a
    winner_metrics = metrics_a

comparison = pd.DataFrame({
    "Metric": [
        "PR-AUC",
        "ROC-AUC",
        "Precision @ 0.5",
        "Recall @ 0.5",
        "F1 @ 0.5",
        "Max F1"
    ],
    "Standard LightGBM": [
        metrics_a["pr_auc"],
        metrics_a["roc_auc"],
        metrics_a["precision_at_0.5"],
        metrics_a["recall_at_0.5"],
        metrics_a["f1_at_0.5"],
        metrics_a["max_f1"]
    ],
    "Focal Loss": [
        metrics_b["pr_auc"],
        metrics_b["roc_auc"],
        metrics_b["precision_at_0.5"],
        metrics_b["recall_at_0.5"],
        metrics_b["f1_at_0.5"],
        metrics_b["max_f1"]
    ]
})

comparison


VERDICT
WINNER: Focal Loss by +0.003289 PR-AUC.


,Metric,Standard LightGBM,Focal Loss
0,PR-AUC,0.405820,0.409109
1,ROC-AUC,0.847339,0.856829
2,Precision @ 0.5,0.418685,0.640845
3,Recall @ 0.5,0.430605,0.323843
4,F1 @ 0.5,0.424561,0.430260
5,Max F1,0.444444,0.446154


## 10. Save Models, Feature Importance and Experiment Results

In [31]:
save_model_outputs(
    model_a,
    preds_a,
    y_calib,
    BASELINE_MODEL_PATH,
    BASELINE_PREDICTION_PATH
)

save_model_outputs(
    model_b,
    preds_b,
    y_calib,
    FOCAL_MODEL_PATH,
    FOCAL_PREDICTION_PATH
)

save_feature_importance(
    model_a,
    feature_columns,
    BASELINE_FEATURE_IMP_PATH
)

save_feature_importance(
    model_b,
    feature_columns,
    FOCAL_FEATURE_IMP_PATH
)

joblib.dump(
    winner_model,
    WINNER_MODEL_PATH
)

pd.DataFrame({
    'actual_fraud': y_calib.to_numpy(),
    'fraud_probability': winner_predictions
}).to_csv(
    WINNER_PREDICTION_PATH,
    index=False
)

save_feature_importance(
    winner_model,
    feature_columns,
    WINNER_FEATURE_IMP_PATH
)

save_preprocessing_artifacts(
    feature_columns,
    categorical_features,
    category_maps,
    winner,
    PREPROCESSING_ARTIFACT_PATH
)

results = {
    'experiment': 'ML Engine V1',
    'primary_metric': 'PR-AUC',
    'training_rows': int(len(X_train)),
    'internal_validation_rows': int(len(X_internal_val)),
    'calibration_rows': int(len(X_calib)),
    'feature_count': int(len(feature_columns)),
    'categorical_feature_count': int(len(categorical_features)),
    'excluded_columns': [
        'isFraud',
        'TransactionID',
        'device_id',
        'ip_address'
    ],
    'internal_validation_fraction': INTERNAL_VALIDATION_FRACTION,
    'focal_parameters': {
        'alpha': FOCAL_ALPHA,
        'gamma': FOCAL_GAMMA
    },
    'winning_model': winner,
    'standard_lightgbm': metrics_a,
    'focal_loss_lightgbm': metrics_b,
    'winning_model_metrics': winner_metrics
}

with open(
    RESULTS_PATH,
    'w'
) as f:
    json.dump(
        results,
        f,
        indent=4
    )

print(
    f"\nExperiment results saved to: {RESULTS_PATH}"
)
print("\nML ENGINE V1 COMPLETE.")


Saving model to: baseline_lightgbm.pkl
Saving predictions to: baseline_m5_predictions.csv

Saving model to: focal_lightgbm.pkl
Saving predictions to: focal_m5_predictions.csv

Saving feature importance to: baseline_feature_importance.csv
                       Feature    Importance
                         card1 217065.853625
                           V70  65737.193264
                         card2  55297.843215
                         addr1  46457.200193
                          V258  37920.351171
                          V317  24194.982042
                            C4  13111.585110
                TransactionAmt  12855.524280
                         id_31  12596.866806
                            D2  11455.143299
          device_txn_last_24hr  10476.652146
                 P_emaildomain  10467.732938
                            C1  10050.187515
            amount_vs_card_avg   9236.537201
                           V30   8839.578604
                            D5   8830.375